In [86]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn. metrics import accuracy_score

In [87]:
df = pd.read_csv("diamonds.csv")

In [88]:
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [89]:
X = df.drop(["price"],axis =1 )
y = df["price"]

In [90]:
X_train, X_test,y_train,y_test = train_test_split(X,y,train_size = 0.75,random_state = 24)

In [91]:
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_train.select_dtypes(include=['object']).columns

In [92]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

In [93]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [94]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

In [95]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
],remainder='drop')


In [96]:
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)


In [97]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
selector = SelectKBest(score_func=f_regression, k=15)

X_train_selected = selector.fit_transform(X_train_prep, y_train)
X_test_selected  = selector.transform(X_test_prep)

In [98]:
model = LinearRegression()
model.fit(X_train_selected, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [99]:
y_pred = model.predict(X_test_selected)

In [100]:
from sklearn.metrics import r2_score

accuracy = r2_score(y_test, y_pred)
print("Model r2_score:", accuracy)

Model r2_score: 0.8943630394181219


In [101]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("feature_selection", SelectKBest(score_func=f_regression)),
    ("model", LogisticRegression(max_iter=1000))
])

In [102]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVC

param_grid = [
    {
        "model": [LinearRegression()],
        "feature_selection__k": [10, 15],
        "model__C": [0.1, 1, 10]
    },
    {
        "model": [RandomForestRegressor()],
        "feature_selection__k": [10, 15],
        "model__n_estimators": [100, 200],
        "model__max_depth": [5, 10]
    }
]

In [103]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

pipe = Pipeline([
    ("preprocess", preprocess),   # ✅ THIS WAS MISSING OR NOT USED
    ("select", SelectKBest(score_func=f_regression)),
    ("model", Ridge())
])

param_grid = {
    "select__k": [5, 10, 15],
    "model__alpha": [0.1, 1, 10]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)   # ✅ RAW X_train IS CORRECT NOW

,estimator,"Pipeline(step...l', Ridge())])"
,param_grid,"{'model__alpha': [0.1, 1, ...], 'select__k': [5, 10, ...]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [104]:
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="r2",        # ✅ regression metric
    n_jobs=-1
)

grid.fit(X_train, y_train)

,estimator,"Pipeline(step...l', Ridge())])"
,param_grid,"{'model__alpha': [0.1, 1, ...], 'select__k': [5, 10, ...]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [105]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

In [106]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

print("Best Params:", grid.best_params_)
print("R2 Score:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAE:", mean_absolute_error(y_test, y_pred))

Best Params: {'model__alpha': 10, 'select__k': 15}
R2 Score: 0.894349744276013
RMSE: 1295.345326934166
MAE: 794.3217976677478


In [107]:
from sklearn.metrics import accuracy_score

# Train accuracy
y_train_pred = best_model.predict(X_train)
train_acc = r2_score(y_train, y_train_pred)

# Test accuracy
y_test_pred = best_model.predict(X_test)
test_acc = r2_score(y_test, y_test_pred)

print("Train Accuracy:", train_acc)
print("Test Accuracy :", test_acc)

Train Accuracy: 0.8945996824858549
Test Accuracy : 0.894349744276013


In [108]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

train_accuracies = []
test_accuracies = []

for train_index, test_index in kf.split(X):
    X_train_k, X_test_k = X.iloc[train_index], X.iloc[test_index]
    y_train_k, y_test_k = y.iloc[train_index], y.iloc[test_index]

    best_model.fit(X_train_k, y_train_k)

    y_train_pred = best_model.predict(X_train_k)
    y_test_pred  = best_model.predict(X_test_k)

    train_accuracies.append(r2_score(y_train_k, y_train_pred))
    test_accuracies.append(r2_score(y_test_k, y_test_pred))

In [109]:
print("Train Accuracies:", train_accuracies)
print("Test Accuracies :", test_accuracies)

print("\nMean Train Accuracy:", np.mean(train_accuracies))
print("Mean Test Accuracy :", np.mean(test_accuracies))

print("\nTrain Accuracy Variance:", np.var(train_accuracies))
print("Test Accuracy Variance :", np.var(test_accuracies))

Train Accuracies: [0.8947198331098376, 0.8936713527010628, 0.8957163139620653, 0.8945892920100685, 0.8932023226740342]
Test Accuracies : [0.8938470481874183, 0.8979973650987777, 0.882123363655497, 0.8930295190491598, 0.9000381286017765]

Mean Train Accuracy: 0.8943798228914137
Mean Test Accuracy : 0.8934070849185259

Train Accuracy Variance: 7.668258829557533e-07
Test Accuracy Variance : 3.8539980333560774e-05


In [110]:
import sklearn
print(sklearn.__version__)

1.7.2


In [ ]:
import pickle


with open("diamond_model.pkl", "wb") as file:
    pickle.dump(best_model, file)

print("Model saved successfully as model.pkl")

Model saved successfully as model.pkl
